
## <span style="background-color: #afe9f4; color: #148c1c; padding: 10px; border-radius: 5px; border-left: 6px solid #399607; display: inline-block; width: 100%;">**Elementary Quantum Probability Applications (Quantum Paradoxes)**</span>

This notebook explains classical decision paradoxes with a quantum-like model:

1. **Conjunction error** — Jane & Bill
2. **Disjunction error** — the Linda problem
3. **Question order effects** — race questions, QQ equality
4. **PD game / disjunction** — Tesar data

The shared core comes from `qmodel_lib.py` (the corrected Layer-1 library). The tools specific to
this notebook (rotation matrix `rotation_matrix`, eq. 4.1; parameter fitting; `qq_value`) are
defined below.

* **Method:** in each application the model parameters (θ₁, θ₂) are *fit* to the observed
 probabilities. The initial state ψ is built from the observed
 marginal of the first question.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import qmodel as q   # corrected Layer-1 library (replaces quantum_lib.py)

np.set_printoptions(precision=3, suppress=True)

## Tools

 `rotation_matrix` (eq. 4.1): rotates the selected coordinate pairs with Givens rotations and
multiplies by an optional sign matrix `D`. `fit_angles`: fits the angles to the observed targets.
`qq_value`: computes the QQ value from the contingency tables of the two orders.

In [3]:
def givens(theta, i, j, n):
    """Plane (i,j) Givens rotation by theta in dimension n."""
    G = np.eye(n, dtype=complex); c, s = np.cos(theta), np.sin(theta)
    G[i, i] = c; G[j, j] = c; G[i, j] = -s; G[j, i] = s
    return G

def rotation_matrix(thetas, planes, dim, D=None):
    """eq. 4.1: V = product of Givens rotations on `planes`; return U = V @ diag(D)."""
    V = np.eye(dim, dtype=complex)
    for th, (i, j) in zip(thetas, planes):
        V = V @ givens(th, i, j, dim)
    return V @ np.diag(np.array(D, dtype=complex)) if D is not None else V

def fit_angles(psi, P_first, yes_idx, targets, D, planes=((0, 4), (1, 3)), dim=5, extra=None):
    """Fit (theta1,theta2) so the model matches `targets`.
    targets = (q_second, q_seq[, q_disjunction]); `extra` adds the disjunction target.
    """
    def model(t):
        U  = rotation_matrix(t, planes, dim, D=D)
        P2 = q.projector_in_basis(U, yes_idx)
        q2   = q.event_prob(P2, psi)
        qseq = q.sequential_prob([P_first, P2], psi)               # first then second
        qdis = 1 - q.sequential_prob([q.complement(P2), q.complement(P_first)], psi)  # 1 - q(~2,~1)
        return q2, qseq, qdis
    def loss(t):
        q2, qseq, qdis = model(t)
        e = (q2 - targets[0])**2 + (qseq - targets[1])**2
        if extra is not None:
            e += (qdis - extra)**2
        return e
    best = None
    for t1 in np.linspace(-np.pi, np.pi, 16):
        for t2 in np.linspace(-np.pi, np.pi, 16):
            r = minimize(loss, [t1, t2], method="Nelder-Mead")
            if best is None or r.fun < best.fun:
                best = r
    return best.x, best.fun

def predict_table(psi, P1, P2):
    """Model joint probabilities for both measurement orders (P1-first and P2-first).
    Returns (order1, order2) dicts keyed by ('y'/'n','y'/'n') -> Table 4.2 / 4.5 style output."""
    n1, n2 = q.complement(P1), q.complement(P2)
    o1 = {('y','y'): q.sequential_prob([P1, P2], psi), ('y','n'): q.sequential_prob([P1, n2], psi),
          ('n','y'): q.sequential_prob([n1, P2], psi), ('n','n'): q.sequential_prob([n1, n2], psi)}
    o2 = {('y','y'): q.sequential_prob([P2, P1], psi), ('y','n'): q.sequential_prob([P2, n1], psi),
          ('n','y'): q.sequential_prob([n2, P1], psi), ('n','n'): q.sequential_prob([n2, n1], psi)}
    return o1, o2

def qq_value(t1, t2):
    """Book convention: qq = (agreement cells)_order1 - (agreement cells)_order2
    = (yy + nn)_1 - (yy + nn)_2.  QQ equality => model qq ~ 0."""
    return (t1[('y','y')] + t1[('n','n')]) - (t2[('y','y')] + t2[('n','n')])


## <span style="background-color: #dbecba; color: #2b35ad; padding: 10px; border-radius: 5px; border-left: 6px solid #399607; display: inline-block; width: 100%;">**Conjunction error — Jane & Bill**</span>

Observed average probabilities (slide 75): **q(J)=0.784** (Jane is an executive),
**q(B)=0.192** (Bill plays jazz), **q(B&J)=0.428**. The more likely event J is evaluated first,
then B. Conjunction error: q(B&J) > q(B). Five-dimensional model; in the J-basis
P_J = diag[1 1 0 0 0].

In [4]:
# initial state in J-basis: marginal q(J)=0.784 (slide hint psi[1]=0.72)
PJ  = q.projector_from_indices([0, 1], 5)
psi = q.normalize([np.sqrt(0.784 - 0.72**2), 0.72, 0.2683, 0.2683, 0.2683])

(t1, t2), loss = fit_angles(psi, PJ, [0, 1], (0.192, 0.428), D=[1, -1, -1, 1, -1])
U  = rotation_matrix([t1, t2], [(0, 4), (1, 3)], 5, D=[1, -1, -1, 1, -1])
PB = q.projector_in_basis(U, [0, 1])

qJ, qB = q.event_prob(PJ, psi), q.event_prob(PB, psi)
qBJ    = q.sequential_prob([PJ, PB], psi)
part   = [q.projector_from_indices([k], 5) for k in range(5)]
IntB   = q.interference(part, PB, psi)

print(f"fit: theta1={t1:.3f}, theta2={t2:.3f}  (loss={loss:.1e})")
print(f"q(J)   = {qJ:.3f}   (empirical 0.784)")
print(f"q(B)   = {qB:.3f}   (empirical 0.192)")
print(f"q(B&J) = {qBJ:.3f}   (empirical 0.428)")
print(f"Conjunction error  q(B&J) > q(B): {qBJ > qB}")
print(f"Interference  q(B) - qT(B) = Int(B) = {IntB:+.3f}   (book: -0.312)")

fit: theta1=2.159, theta2=-0.614  (loss=3.5e-12)
q(J)   = 0.784   (empirical 0.784)
q(B)   = 0.192   (empirical 0.192)
q(B&J) = 0.428   (empirical 0.428)
Conjunction error  q(B&J) > q(B): True
Interference  q(B) - qT(B) = Int(B) = -0.310   (book: -0.312)


In [5]:
# Examples: total-probability decomposition and interference for event B
# qT(B) = q(J,B) + q(~J,B);  interference = q(B) - qT(B)
PnotJ  = q.complement(PJ)
qJB    = q.sequential_prob([PJ, PB], psi)      # q(J, B)  = conjunction q(B&J)
qnotJB = q.sequential_prob([PnotJ, PB], psi)   # q(~J, B)
qT_B   = qJB + qnotJB                          # total probability of B over {J, ~J}

print(f"q(J, B)  = {qJB:.3f}   (= q(B&J), the conjunction)")
print(f"q(~J, B) = {qnotJB:.3f}")
print(f"qT(B) = q(J,B) + q(~J,B) = {qT_B:.3f}")
print(f"q(B)  = {qB:.3f}")
print(f"Interference  q(B) - qT(B) = {qB - qT_B:+.3f}")

q(J, B)  = 0.428   (= q(B&J), the conjunction)
q(~J, B) = 0.074
qT(B) = q(J,B) + q(~J,B) = 0.502
q(B)  = 0.192
Interference  q(B) - qT(B) = -0.310



## <span style="background-color: #ddf0bc; color: #2b35ad; padding: 10px; border-radius: 5px; border-left: 6px solid #399607; display: inline-block; width: 100%;">**Disjunction error — the Linda problem**</span>

Observed probabilities (slide 84): **q(F)=0.83** (feminist), **q(B)=0.26** (bank teller),
**q(F&B)=0.36**, **q(F or B)=0.60**. Two fallacies at once:
conjunction q(F&B) > q(B) **and** disjunction q(F or B) < q(F).
The disjunction is computed with ¬B measured first (slide 88): q(F or B) = 1 − q(¬B, ¬F).

* Since two angles are fit to three targets, a small residual remains; both paradoxes still hold
 qualitatively. For a tighter fit you can open an extra parameter (e.g. the D signs).

In [6]:
PF   = q.projector_from_indices([0, 1], 5)
psiL = q.normalize([np.sqrt(0.83 - 0.75**2), 0.75, 0.238, 0.238, 0.238])

# fit theta to the conjunction (0.36) AND disjunction (0.60) targets
(s1, s2), lossL = fit_angles(psiL, PF, [0, 1], (0.26, 0.36), D=[1, -1, -1, 1, -1], extra=0.60)
UL  = rotation_matrix([s1, s2], [(0, 4), (1, 3)], 5, D=[1, -1, -1, 1, -1])
PBt = q.projector_in_basis(UL, [0, 1])

qF, qBt = q.event_prob(PF, psiL), q.event_prob(PBt, psiL)
qFB     = q.sequential_prob([PF, PBt], psiL)
qForB   = 1 - q.sequential_prob([q.complement(PBt), q.complement(PF)], psiL)

print(f"fit: theta=({s1:.3f},{s2:.3f})  loss={lossL:.1e}")
print(f"q(F)      = {qF:.3f}  (emp 0.83)")
print(f"q(B)      = {qBt:.3f}  (emp 0.26)")
print(f"q(F&B)    = {qFB:.3f}  (emp 0.36)   conjunction error q(F&B)>q(B): {qFB > qBt}")
print(f"q(F or B) = {qForB:.3f}  (emp 0.60)   disjunction error q(F or B)<q(F): {qForB < qF}")

fit: theta=(1.922,-0.625)  loss=5.0e-03
q(F)      = 0.830  (emp 0.83)
q(B)      = 0.222  (emp 0.26)
q(F&B)    = 0.401  (emp 0.36)   conjunction error q(F&B)>q(B): True
q(F or B) = 0.643  (emp 0.60)   disjunction error q(F or B)<q(F): True



## <span style="background-color: #95b4bc; color: #2b35ad; padding: 10px; border-radius: 5px; border-left: 6px solid #193387; display: inline-block; width: 100%;">**Question order effects — QQ equality**</span>

Two binary questions (W: do whites dislike blacks; B: do blacks dislike whites) are asked in both
orders. The quantum model yields the **QQ equality**, a parameter-free prediction: the
"disagreement" probabilities are equal across the two orders, i.e. model qq ≈ 0.
Parameters from the slides: θ₁=0.900, θ₂=0.715, D=diag[1 1 1 −1 −1], P_W=diag[1 1 1 0 0].

 * **Data:** real values from Table 4.1 of QCD2-BB are used below. The model reproduces the
 parameter-free **QQ equality** (model qq ≈ 0). The full predicted joint table (Table 4.2)
 depends on the book's exact initial state, so the joints here are illustrative.

In [7]:
# --- Table 4.1 (QCD2-BB), real values ---
WB = {('y','y'): 0.399, ('y','n'): 0.017, ('n','y'): 0.161, ('n','n'): 0.423}  # W asked first
BW = {('y','y'): 0.401, ('y','n'): 0.138, ('n','y'): 0.060, ('n','n'): 0.401}  # B asked first
print(f"Empirical qq (Table 4.1) = {qq_value(WB, BW):+.3f}")

# model: initial state matches the act-first W marginal q(Wy) = 0.416
PW   = q.projector_from_indices([0, 1, 2], 5)                # diag[1 1 1 0 0]
psiW = q.normalize([np.sqrt(0.416/3)]*3 + [np.sqrt(0.584/2)]*2)
Uwb  = rotation_matrix([0.900, 0.715], [(0, 4), (1, 3)], 5, D=[1, 1, 1, -1, -1])
PBw  = q.projector_in_basis(Uwb, [0, 1, 2])

mWB, mBW = predict_table(psiW, PW, PBw)   # Table 4.2-style predicted joints
print(f"q(Wy) = {q.event_prob(PW, psiW):.3f}  (target 0.416)")
print("model WB order:", {k: round(v, 3) for k, v in mWB.items()})
print("model BW order:", {k: round(v, 3) for k, v in mBW.items()})
print(f"Model qq = {qq_value(mWB, mBW):+.4f}   (QQ equality => ~0)")

Empirical qq (Table 4.1) = +0.020
q(Wy) = 0.416  (target 0.416)
model WB order: {('y', 'y'): 0.271, ('y', 'n'): 0.145, ('n', 'y'): 0.305, ('n', 'n'): 0.279}
model BW order: {('y', 'y'): 0.535, ('y', 'n'): 0.437, ('n', 'y'): 0.013, ('n', 'n'): 0.016}
Model qq = -0.0000   (QQ equality => ~0)



## <span style="background-color: #78e6e2; color: #2b35ad; padding: 10px; border-radius: 5px; border-left: 6px solid #0c5b1d; display: inline-block; width: 100%;">**PD game / disjunction effect — Tesar data**</span>

Prisoner's dilemma: does the player cooperate after predicting the opponent's move, or without
predicting it? The unitary is a product of three, U = D·V·E (θ₁=0.900, θ₂=0.551,
D=diag[1 −1 −1 1 −1], E=diag[−1 1 −1 −1 −1]). Disjunction effect: the gap between q(C′) and
q_T(C′) = q(C,C′)+q(D,C′) (interference).

> **Data:** real Table 4.4 (Tesar 2020) values are used; empirical qq = −0.05 (matches the book).
> The model shows the QQ equality (qq ≈ 0) and a disjunction interference effect; the exact
> magnitude/sign depends on the book's initial state.

In [8]:
# --- Table 4.4 (Tesar 2020), real values.  y = C / C', n = D / D' ---
ACT = {('y','y'): 0.40, ('y','n'): 0.25, ('n','y'): 0.12, ('n','n'): 0.23}  # Act First   (N=77)
PRE = {('y','y'): 0.25, ('y','n'): 0.17, ('n','y'): 0.15, ('n','n'): 0.43}  # Predict First (N=69)
print(f"Empirical qq (Table 4.4) = {qq_value(ACT, PRE):+.3f}   (book -0.05)")

# model: initial state matches the act-first cooperation marginal q(C) = 0.65
PC   = q.projector_from_indices([0, 1, 2], 5)
psiC = q.normalize([np.sqrt(0.65/3)]*3 + [np.sqrt(0.35/2)]*2)
V    = rotation_matrix([0.900, 0.551], [(0, 4), (1, 3)], 5)
D    = np.diag(np.array([1, -1, -1, 1, -1], dtype=complex))
E    = np.diag(np.array([-1, 1, -1, -1, -1], dtype=complex))
Upd  = D @ V @ E
PCp  = q.projector_in_basis(Upd, [0, 1, 2])

qC, qCp = q.event_prob(PC, psiC), q.event_prob(PCp, psiC)
qTC = q.sequential_prob([PC, PCp], psiC) + q.sequential_prob([q.complement(PC), PCp], psiC)
mACT, mPRE = predict_table(psiC, PC, PCp)
print(f"U unitary: {q.is_unitary(Upd)}")
print(f"q(C) = {qC:.3f} (target 0.65)   q(C') = {qCp:.3f}   qT(C') = {qTC:.3f}")
print(f"Disjunction interference  q(C') - qT(C') = {qCp - qTC:+.3f}")
print(f"Model qq = {qq_value(mACT, mPRE):+.4f}   (QQ equality => ~0)")

Empirical qq (Table 4.4) = -0.050   (book -0.05)
U unitary: True
q(C) = 0.650 (target 0.65)   q(C') = 0.250   qT(C') = 0.613
Disjunction interference  q(C') - qT(C') = -0.363
Model qq = +0.0000   (QQ equality => ~0)
